In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load dataset
df = pd.read_csv("WELFake_Dataset.csv")

# Drop first column (ID column)
df = df.iloc[:, 1:]

# Drop missing values
df = df.dropna()

# Combine title and text
df["content"] = df["title"] + " " + df["text"]

X = df["content"]

# Flip labels: 0 -> real, 1 -> fake
# Original: 0=fake, 1=real
y = df["label"].apply(lambda x: 1 - x)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorization
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.7,
    min_df=2,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train Linear SVM
# LinearSVC is fast but has no predict_proba -> calibrate to get probabilities
base_svm = LinearSVC(class_weight="balanced", random_state=42)
svm = CalibratedClassifierCV(base_svm, method="sigmoid", cv=3)  # cv=3 keeps it fast

svm.fit(X_train_tfidf, y_train)

# Evaluate model on test set
y_pred = svm.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Real (0)", "Fake (1)"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Evaluate on training set
y_train_pred = svm.predict(X_train_tfidf)

print("\nTRAINING SET PERFORMANCE")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print(classification_report(y_train, y_train_pred, target_names=["Real (0)", "Fake (1)"]))

print("\nTEST SET PERFORMANCE")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Real (0)", "Fake (1)"]))

print("\nTRAIN confusion matrix:")
print(confusion_matrix(y_train, y_train_pred))

print("\nTEST confusion matrix:")
print(confusion_matrix(y_test, y_pred))

#exec time 1.02

Accuracy: 0.9719038300251608
              precision    recall  f1-score   support

    Real (0)       0.97      0.97      0.97      7302
    Fake (1)       0.97      0.97      0.97      7006

    accuracy                           0.97     14308
   macro avg       0.97      0.97      0.97     14308
weighted avg       0.97      0.97      0.97     14308

Confusion Matrix:
[[7110  192]
 [ 210 6796]]

TRAINING SET PERFORMANCE
Accuracy: 0.999178738052386
              precision    recall  f1-score   support

    Real (0)       1.00      1.00      1.00     29207
    Fake (1)       1.00      1.00      1.00     28022

    accuracy                           1.00     57229
   macro avg       1.00      1.00      1.00     57229
weighted avg       1.00      1.00      1.00     57229


TEST SET PERFORMANCE
Accuracy: 0.9719038300251608
              precision    recall  f1-score   support

    Real (0)       0.97      0.97      0.97      7302
    Fake (1)       0.97      0.97      0.97      7006

   

In [3]:
def test_article(title, text, model, vectorizer, crisis=False):
    content = title + " " + text
    content_tfidf = vectorizer.transform([content])

    probs = model.predict_proba(content_tfidf)[0]
    class_to_prob = dict(zip(model.classes_, probs))

    prob_fake = class_to_prob.get(0, None)
    prob_real = class_to_prob.get(1, None)
    if prob_fake is None or prob_real is None:
        raise ValueError(f"Expected classes [0, 1], but got {model.classes_}")

    threshold = 0.8 if crisis else 0.5
    classification = "Most likely real" if prob_real >= threshold else "Most likely fake"
    return classification, prob_real, prob_fake

title = "Kim Jong Un chooses teen daughter as heir, says Seoul"
text = "North Korean leader Kim Jong Un has selected his daughter as his heir, South Korea's spy agency told lawmakers on Thursday.Little is known about Kim Ju Ae, who in recent months has been pictured beside her father in high-profile events like a visit to Beijing in September- her first known trip abroad.The National Intelligence Service (NIS) said it took a range of circumstances into account including her increasingly prominent public presence at official events in making this assessment."

result, prob_real, prob_fake = test_article(
    title=title,
    text=text,
    model=svm,           # <-- Linear SVM (calibrated)
    vectorizer=vectorizer,
    crisis=True
)

print("Classification:", result)
print("Probability real:", round(prob_real, 3))
print("Probability fake:", round(prob_fake, 3))


Classification: Most likely real
Probability real: 0.815
Probability fake: 0.185
